### 🛠️ Initialize Notebook Variables

**Only modify entries under _USER CONFIGURATION_.**

In [ ]:
import json
import utils
from typing import List

from apimtypes import API, APIM_SKU, APIOperation, GET_APIOperation, HTTP_VERB, INFRASTRUCTURE, NamedValue, PolicyFragment, Region
from console import print_error, print_ok
from azure_resources import get_infra_rg_name

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = Region.EAST_US_2
index       = 1
apim_sku    = APIM_SKU.BASICV2              # Options: 'DEVELOPER', 'BASIC', 'STANDARD', 'PREMIUM', 'BASICV2', 'STANDARDV2', 'PREMIUMV2'
deployment  = INFRASTRUCTURE.SIMPLE_APIM    # Options: see supported_infras below
api_prefix  = 'cors-'                       # ENTER A PREFIX FOR THE APIS TO REDUCE COLLISION POTENTIAL WITH OTHER SAMPLES
tags        = ['dynamic-cors']              # ENTER DESCRIPTIVE TAGS



# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder    = 'dynamic-cors'
rg_name          = get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.AFD_APIM_PE,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
    INFRASTRUCTURE.SIMPLE_APIM
]
nb_helper        = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index = index,
    apim_sku = apim_sku
)

# API paths (2 per phase, all coexisting side-by-side)
bl_products_path   = f'{api_prefix}bl-products'
bl_analytics_path  = f'{api_prefix}bl-analytics'
ph1_products_path  = f'{api_prefix}ph1-products'
ph1_analytics_path = f'{api_prefix}ph1-analytics'
ph2_products_path  = f'{api_prefix}ph2-products'
ph2_analytics_path = f'{api_prefix}ph2-analytics'
ph3_products_path  = f'{api_prefix}ph3-products'
ph3_analytics_path = f'{api_prefix}ph3-analytics'
ph3_admin_path     = f'{api_prefix}admin'

# Cache key used for CORS origin mapping in Phase 3
cors_cache_key = 'corsOriginMapping'

# Allowed origins per API team
products_origins  = ['https://shop.contoso.com', 'https://admin.contoso.com']
analytics_origins = ['https://dashboard.contoso.com']

# JSON mapping for the Named Value (Phase 2 API IDs only)
cors_origin_mapping = {
    ph2_products_path:  products_origins,
    ph2_analytics_path: analytics_origins,
}
cors_origin_mapping_json = json.dumps(cors_origin_mapping, separators=(',', ':'))

# JSON mapping for the cache (Phase 3 API IDs)
cors_cache_mapping = {
    ph3_products_path:  products_origins,
    ph3_analytics_path: analytics_origins,
}
cors_cache_mapping_json = json.dumps(cors_cache_mapping, separators=(',', ':'))

# Load operation policies
pol_cors_get = utils.read_policy_xml('cors-get.xml', sample_name = sample_folder)

# Shared operations for all CORS demo APIs
cors_get_op     = GET_APIOperation('Returns mock response with CORS status', pol_cors_get)
cors_options_op = APIOperation('OPTIONS', 'OPTIONS', '/', HTTP_VERB.OPTIONS, 'CORS preflight request')


# ---- Baseline: Native APIM <cors> policy ----

pol_baseline = utils.read_policy_xml('cors-baseline-native.xml', sample_name = sample_folder)

apis_baseline: List[API] = [
    API(bl_products_path, 'Baseline Products', bl_products_path,
        'Products API with native <cors> policy (static origins)',
        pol_baseline, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(bl_analytics_path, 'Baseline Analytics', bl_analytics_path,
        'Analytics API with native <cors> policy (static origins)',
        pol_baseline, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Phase 1: Hard-coded fragment ----

pf_hardcoded_xml = utils.read_policy_xml('pf-dynamic-cors-hardcoded.xml', sample_name = sample_folder)
pf_hardcoded_xml = pf_hardcoded_xml.replace('{products_api_id}', ph1_products_path)
pf_hardcoded_xml = pf_hardcoded_xml.replace('{analytics_api_id}', ph1_analytics_path)

pol_api_ph1 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_ph1 = pol_api_ph1.replace('{fragment_id}', 'DynamicCorsHardcoded')

apis_ph1: List[API] = [
    API(ph1_products_path, 'Phase 1 Products', ph1_products_path,
        'Products API with hard-coded CORS fragment',
        pol_api_ph1, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(ph1_analytics_path, 'Phase 1 Analytics', ph1_analytics_path,
        'Analytics API with hard-coded CORS fragment',
        pol_api_ph1, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Phase 2: Named Values fragment ----

pf_nv_xml = utils.read_policy_xml('pf-dynamic-cors-named-values.xml', sample_name = sample_folder)
pf_nv_xml = pf_nv_xml.replace('{cors_origin_mapping}', '{{CorsOriginMapping}}')

pol_api_ph2 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_ph2 = pol_api_ph2.replace('{fragment_id}', 'DynamicCorsNamedValues')

apis_ph2: List[API] = [
    API(ph2_products_path, 'Phase 2 Products', ph2_products_path,
        'Products API with Named Values CORS fragment',
        pol_api_ph2, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(ph2_analytics_path, 'Phase 2 Analytics', ph2_analytics_path,
        'Analytics API with Named Values CORS fragment',
        pol_api_ph2, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Phase 3: Cache-backed fragment with admin API ----

pf_cached_xml = utils.read_policy_xml('pf-dynamic-cors-cached.xml', sample_name = sample_folder)

pol_api_ph3 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_ph3 = pol_api_ph3.replace('{fragment_id}', 'DynamicCorsCached')

pol_admin_load  = utils.read_policy_xml('admin-load-cache.xml', sample_name = sample_folder)
pol_admin_clear = utils.read_policy_xml('admin-clear-cache.xml', sample_name = sample_folder)

cache_key_param = [{'name': 'cacheKey', 'required': True, 'type': 'string', 'description': 'The cache key to operate on'}]

apis_ph3: List[API] = [
    API(ph3_products_path, 'Phase 3 Products', ph3_products_path,
        'Products API with cache-backed CORS fragment',
        pol_api_ph3, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(ph3_analytics_path, 'Phase 3 Analytics', ph3_analytics_path,
        'Analytics API with cache-backed CORS fragment',
        pol_api_ph3, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]

api_admin: API = API(
    ph3_admin_path, 'Phase 3 Admin', ph3_admin_path,
    'Admin API for managing APIM cache entries',
    operations = [
        APIOperation('load-cache', 'Load Cache', '/load-cache/{cacheKey}', HTTP_VERB.POST,
                     'Store a value in the APIM internal cache', pol_admin_load, cache_key_param),
        APIOperation('clear-cache', 'Clear Cache', '/clear-cache/{cacheKey}', HTTP_VERB.POST,
                     'Remove a value from the APIM internal cache', pol_admin_clear, cache_key_param),
    ],
    tags = tags,
    subscriptionRequired = True,
)


# ---- Shared: Policy fragments and Named Values ----

pfs: List[PolicyFragment] = [
    PolicyFragment('DynamicCorsHardcoded', pf_hardcoded_xml, 'Dynamic CORS - hard-coded API-to-origins mapping.'),
    PolicyFragment('DynamicCorsNamedValues', pf_nv_xml, 'Dynamic CORS - origins loaded from a Named Value.'),
    PolicyFragment('DynamicCorsCached', pf_cached_xml, 'Dynamic CORS - origins loaded from APIM internal cache.'),
]

nvs: List[NamedValue] = [
    NamedValue('CorsOriginMapping', cors_origin_mapping_json, False),
]

# Combine all APIs for a single deployment
all_apis = apis_baseline + apis_ph1 + apis_ph2 + apis_ph3 + [api_admin]


print_ok('Notebook initialized')

### 🚀 Deploy All Phases

Deploys all nine APIs side-by-side in a single operation alongside all three policy fragments and the Named Value:

| Phase | APIs | CORS Mechanism |
| ----- | ---- | -------------- |
| **Baseline** | `cors-bl-products`, `cors-bl-analytics` | Native `<cors>` policy with static origins |
| **Phase 1** | `cors-ph1-products`, `cors-ph1-analytics` | `DynamicCorsHardcoded` policy fragment |
| **Phase 2** | `cors-ph2-products`, `cors-ph2-analytics` | `DynamicCorsNamedValues` policy fragment |
| **Phase 3** | `cors-ph3-products`, `cors-ph3-analytics`, `cors-admin` | `DynamicCorsCached` policy fragment + admin API to load cache |

This lets you inspect and compare all four approaches without redeployment.

In [ ]:
# Deploy all APIs across all phases, plus policy fragments and Named Values
bicep_parameters = {
    'apis'            : {'value': [api.to_dict() for api in all_apis]},
    'policyFragments' : {'value': [pf.to_dict() for pf in pfs]},
    'namedValues'     : {'value': [nv.to_dict() for nv in nvs]},
}

output = nb_helper.deploy_sample(bicep_parameters)

if output.success:
    apim_name        = output.get('apimServiceName', 'APIM Service Name')
    apim_gateway_url = output.get('apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_apis        = output.getJson('apiOutputs', 'APIs')

    # Extract the admin API subscription key for Phase 3
    admin_subscription_key = None
    for api_output in apim_apis:
        if api_output.get('name') == ph3_admin_path:
            admin_subscription_key = api_output.get('subscriptionPrimaryKey')
            break

    print_ok('Deployment completed successfully - all 9 APIs deployed side-by-side')
else:
    print_error('Deployment failed!')
    raise SystemExit(1)

### ✅ Test Baseline: Native APIM `<cors>` Policy

The baseline APIs use APIM's built-in `<cors>` policy with a static origin list: `https://shop.contoso.com` and `https://admin.contoso.com`.

| API | Origin | Expected Result |
| --- | ------ | --------------- |
| **Baseline Products** | `https://shop.contoso.com` | PASS - origin is in the static list |
| **Baseline Products** | `https://admin.contoso.com` | PASS - origin is in the static list |
| **Baseline Products** | `https://unauthorized.contoso.net` | PASS (rejected) - origin is not in the list |
| **Baseline Analytics** | `https://dashboard.contoso.com` | **FAIL** - the dashboard origin is not in the static list |

This demonstrates the core limitation: the native `<cors>` policy cannot provide **per-API origin control**.

In [ ]:
import json
import socket
import time
import requests as http_requests
from urllib.parse import urlparse
from apimtesting import ApimTesting

if 'apim_gateway_url' not in locals():
    raise SystemExit(1)


# ---- Gateway readiness check (DNS propagation after new APIM instances) ----

gateway_host = urlparse(apim_gateway_url).hostname
max_dns_wait = 120
poll_interval = 10

for elapsed in range(0, max_dns_wait + 1, poll_interval):
    try:
        socket.getaddrinfo(gateway_host, 443)
        if elapsed:
            print_ok(f'Gateway DNS resolved after {elapsed}s')
        else:
            print_ok(f'Gateway DNS resolved: {gateway_host}')
        break
    except socket.gaierror:
        if not elapsed:
            print(f'Waiting for APIM gateway DNS to propagate ({gateway_host}) ...')
        else:
            print(f'  Still waiting ... ({elapsed}s / {max_dns_wait}s)')
        time.sleep(poll_interval)
else:
    print_error(f'Gateway DNS did not resolve within {max_dns_wait}s.')
    print_error('If APIM was just provisioned, wait a few more minutes and re-run this cell.')
    raise SystemExit(1)


# ---- Shared HTTP session (reuses TCP+TLS connections across all tests) ----

endpoint_url, request_headers, allow_insecure_tls = utils.get_endpoint(
    deployment, rg_name, apim_gateway_url
)

http_session = http_requests.Session()
http_session.verify = not allow_insecure_tls
if request_headers:
    http_session.headers.update(request_headers)


# ---- Helper: send an OPTIONS preflight request ----

def cors_options_request(base_url, path, origin):
    """Send an OPTIONS preflight request and return (status_code, response_headers, elapsed_ms)."""
    url = f'{base_url}/{path}'
    resp = http_session.options(
        url, headers={'Origin': origin, 'Access-Control-Request-Method': 'GET'}, timeout=30
    )
    elapsed_ms = round(resp.elapsed.total_seconds() * 1000, 1)
    return resp.status_code, dict(resp.headers), elapsed_ms


# ---- Helper: track individual test results ----

test_results = []

def track(suite, phase, label, actual, expected, duration_ms=None):
    """Call verify and record the per-test result for the summary table."""
    passed = suite.verify(actual, expected, label)
    test_results.append({
        'Phase': phase,
        'Test': label,
        'Expected': str(expected),
        'Actual': str(actual),
        'Duration (ms)': duration_ms,
        'Result': passed,
    })
    return passed


# ---- Warm-up: establish the HTTP connection before scenario tests ----

tests_warmup = ApimTesting(
    'Dynamic CORS - Warm-up Test', sample_folder, deployment
)

warmup_url = f'{endpoint_url}/{bl_products_path}'
warmup_resp = http_session.get(
    warmup_url,
    headers={'Origin': 'https://shop.contoso.com'},
    timeout=30,
)
warmup_ms = round(warmup_resp.elapsed.total_seconds() * 1000, 1)
track(
    tests_warmup, 'Warm-up',
    'Gateway warm-up GET - establish reusable HTTP session',
    warmup_resp.status_code, 200, warmup_ms,
)
tests_warmup.print_summary()

print_ok('HTTP session warm-up complete')


# ---- Helper: run the shared per-API CORS test matrix (Phases 1, 2 & 3) ----

def run_phase_tests(suite, phase, products_path, analytics_path):
    """Execute the full OPTIONS + GET CORS test matrix for a phase."""
    pfx = phase.upper().replace(' ', '')

    # ---- OPTIONS preflight tests ----

    status, hdrs, ms = cors_options_request(
        endpoint_url, products_path, 'https://shop.contoso.com'
    )
    track(suite, phase, f'{pfx} Products OPTIONS - allowed origin (shop)', status, 200, ms)
    track(
        suite, phase, f'{pfx} Products Access-Control-Allow-Origin (shop)',
        hdrs.get('Access-Control-Allow-Origin', ''),
        'https://shop.contoso.com'
    )

    status, hdrs, ms = cors_options_request(
        endpoint_url, products_path, 'https://admin.contoso.com'
    )
    track(suite, phase, f'{pfx} Products OPTIONS - allowed origin (admin)', status, 200, ms)
    track(
        suite, phase, f'{pfx} Products Access-Control-Allow-Origin (admin)',
        hdrs.get('Access-Control-Allow-Origin', ''),
        'https://admin.contoso.com'
    )

    status, _, ms = cors_options_request(
        endpoint_url, products_path, 'https://unauthorized.contoso.net'
    )
    track(suite, phase, f'{pfx} Products OPTIONS - unauthorized origin', status, 403, ms)

    status, hdrs, ms = cors_options_request(
        endpoint_url, analytics_path, 'https://dashboard.contoso.com'
    )
    track(
        suite, phase,
        f'{pfx} Analytics OPTIONS - allowed origin', status, 200, ms
    )
    track(
        suite, phase, f'{pfx} Analytics Access-Control-Allow-Origin',
        hdrs.get('Access-Control-Allow-Origin', ''),
        'https://dashboard.contoso.com'
    )

    status, _, ms = cors_options_request(
        endpoint_url, analytics_path, 'https://shop.contoso.com'
    )
    track(
        suite, phase,
        f'{pfx} Analytics OPTIONS - unauthorized origin (shop)',
        status, 403, ms
    )

    # ---- GET tests (body includes corsAllowed) ----

    def session_get(path, origin, label):
        """GET via the shared session and return (body_dict, elapsed_ms)."""
        url = f'{endpoint_url}/{path}'
        resp = http_session.get(url, headers={'Origin': origin}, timeout=30)
        ms = round(resp.elapsed.total_seconds() * 1000, 1)
        body = resp.json() if resp.ok else {}
        print(f'  {label}: {resp.status_code} ({ms} ms)')
        return body, ms

    body, ms = session_get(
        products_path, 'https://shop.contoso.com',
        f'{pfx} Products GET (shop)'
    )
    track(
        suite, phase, f'{pfx} Products GET corsAllowed (shop)',
        body.get('corsAllowed'), True, ms
    )
    track(
        suite, phase, f'{pfx} Products GET allowedOrigin (shop)',
        body.get('allowedOrigin'), 'https://shop.contoso.com'
    )

    body, ms = session_get(
        products_path, 'https://unauthorized.contoso.net',
        f'{pfx} Products GET (unauthorized)'
    )
    track(
        suite, phase, f'{pfx} Products GET corsAllowed (unauthorized)',
        body.get('corsAllowed'), False, ms
    )

    body, ms = session_get(
        analytics_path, 'https://dashboard.contoso.com',
        f'{pfx} Analytics GET (dashboard)'
    )
    track(
        suite, phase, f'{pfx} Analytics GET corsAllowed',
        body.get('corsAllowed'), True, ms
    )

    body, ms = session_get(
        analytics_path, 'https://shop.contoso.com',
        f'{pfx} Analytics GET (unauthorized)'
    )
    track(
        suite, phase, f'{pfx} Analytics GET corsAllowed (unauthorized)',
        body.get('corsAllowed'), False, ms
    )


# ---- Baseline tests (unique structure - no GET tests, different expectations) ----

tests_baseline = ApimTesting(
    'Dynamic CORS - Baseline Tests', sample_folder, deployment
)

BL = 'Baseline'

# Products - allowed origin (shop)
status, hdrs, ms = cors_options_request(
    endpoint_url, bl_products_path, 'https://shop.contoso.com'
)
track(tests_baseline, BL, 'BL Products OPTIONS - shop origin (in static list)', status, 200, ms)
track(
    tests_baseline, BL, 'BL Products Access-Control-Allow-Origin matches shop origin',
    hdrs.get('Access-Control-Allow-Origin', ''), 'https://shop.contoso.com'
)

# Products - allowed origin (admin)
status, hdrs, ms = cors_options_request(
    endpoint_url, bl_products_path, 'https://admin.contoso.com'
)
track(tests_baseline, BL, 'BL Products OPTIONS - admin origin (in static list)', status, 200, ms)
track(
    tests_baseline, BL, 'BL Products Access-Control-Allow-Origin matches admin origin',
    hdrs.get('Access-Control-Allow-Origin', ''), 'https://admin.contoso.com'
)

# Products - unauthorized origin (correctly rejected)
status, hdrs, ms = cors_options_request(
    endpoint_url, bl_products_path, 'https://unauthorized.contoso.net'
)
has_acao = 'Access-Control-Allow-Origin' in hdrs
track(
    tests_baseline, BL,
    'BL Products OPTIONS - unauthorized origin has no Access-Control-Allow-Origin',
    has_acao, False, ms
)

# Analytics - dashboard origin NOT in the static list
status, hdrs, ms = cors_options_request(
    endpoint_url, bl_analytics_path, 'https://dashboard.contoso.com'
)
has_acao = 'Access-Control-Allow-Origin' in hdrs
track(
    tests_baseline, BL,
    'BL Analytics OPTIONS - dashboard origin MISSING from static list'
    ' (no Access-Control-Allow-Origin = EXPECTED LIMITATION)',
    has_acao, False, ms
)

tests_baseline.print_summary()

print_ok('Baseline tests complete - limitations of native <cors> demonstrated')

### ✅ Test Phase 1: Hard-coded CORS Fragment

The Phase 1 APIs use the `DynamicCorsHardcoded` policy fragment with per-API origin mapping embedded in C#. Both APIs now get the correct CORS behaviour for their specific frontends.

| API | Origin | Expected Result |
| --- | ------ | --------------- |
| **Phase 1 Products** | `https://shop.contoso.com` | PASS - mapped origin |
| **Phase 1 Products** | `https://admin.contoso.com` | PASS - mapped origin |
| **Phase 1 Products** | `https://unauthorized.contoso.net` | PASS (rejected) - 403 |
| **Phase 1 Analytics** | `https://dashboard.contoso.com` | PASS - per-API mapping works (was blocked in baseline) |
| **Phase 1 Analytics** | `https://shop.contoso.com` | PASS (rejected) - 403, shop is not an Analytics origin |

In [ ]:
from apimtesting import ApimTesting

if 'apim_gateway_url' not in locals():
    raise SystemExit(1)

tests_ph1 = ApimTesting(
    'Dynamic CORS - Phase 1 Tests', sample_folder, deployment
)

run_phase_tests(tests_ph1, 'Phase 1', ph1_products_path, ph1_analytics_path)
tests_ph1.print_summary()

print_ok('Phase 1 tests complete - per-API CORS verified via hard-coded fragment')

### ✅ Test Phase 2: Named Values CORS Fragment

The Phase 2 APIs use the `DynamicCorsNamedValues` policy fragment, which reads the JSON origin mapping from the `CorsOriginMapping` Named Value. All tests should match Phase 1 since the mapping data is identical.

**Advantage over Phase 1:** Origins can be updated in the Azure portal or via CLI without redeploying the policy fragment. **Limitation:** APIM Named Values have a **4,096-character limit** per value.

In [ ]:
from apimtesting import ApimTesting

if 'apim_gateway_url' not in locals():
    raise SystemExit(1)

tests_ph2 = ApimTesting(
    'Dynamic CORS - Phase 2 Tests', sample_folder, deployment
)

run_phase_tests(tests_ph2, 'Phase 2', ph2_products_path, ph2_analytics_path)
tests_ph2.print_summary()

print_ok('Phase 2 tests complete - per-API CORS verified via Named Values fragment')

### ✅ Test Phase 3: Cache-backed CORS Fragment

Phase 3 uses the APIM internal cache to store origin mappings. The cache must be loaded via the admin API before any CORS evaluation can succeed.

**Step 1 - Clear the cache:** Call `POST /cors-admin/clear-cache/corsOriginMapping` to ensure any stale cache entry from a previous run is removed.

**Step 2 - Verify fail-closed behaviour:** With the cache empty, both Phase 3 APIs must return `503 Service Unavailable`.

**Step 3 - Load the cache:** Call `POST /cors-admin/load-cache/corsOriginMapping` with the JSON origin mapping.

**Step 4 - Run CORS tests:** After the cache is loaded, Phase 3 should behave identically to Phases 1 and 2.

> **Note:** The APIM internal cache is shared across the entire instance. The TTL is set to effectively infinite (`2147483647` seconds). Re-calling the load endpoint replaces the previous cache entry. For isolated, high-scale, or multi-region deployments, an external Azure Cache for Redis can be used instead.

> **Security:** The admin API is protected by a subscription key only for simplicity in this lab. In production, add `validate-azure-ad-token` or `validate-jwt` to the admin API's inbound policy. See the [authX](../authX/) and [authX-pro](../authX-pro/) samples for patterns.

In [ ]:
import json
from apimtesting import ApimTesting

if 'apim_gateway_url' not in locals():
    raise SystemExit(1)

tests_ph3 = ApimTesting(
    'Dynamic CORS - Phase 3 Tests', sample_folder, deployment
)

admin_base_headers = {'api-key': admin_subscription_key}


# ---- Step 1: Clear the cache (ensures a clean slate even on re-runs) ----

print('Step 1: Clearing CORS cache via admin API ...\n')

clear_url = f'{endpoint_url}/{ph3_admin_path}/clear-cache/{cors_cache_key}'
clear_resp = http_session.post(
    clear_url,
    headers = admin_base_headers,
    timeout = 30,
)
clear_ms = round(clear_resp.elapsed.total_seconds() * 1000, 1)

track(
    tests_ph3, 'Phase 3', 'PH3 Admin POST /clear-cache - 200 OK',
    clear_resp.status_code, 200, clear_ms
)

print(f'  Clear response: {clear_resp.text}')


# ---- Step 2: Verify fail-closed behaviour (cache is empty) ----

print('\nStep 2: Verifying 503 when CORS cache is not initialized ...\n')

status, hdrs, ms = cors_options_request(
    endpoint_url, ph3_products_path, 'https://shop.contoso.com'
)
track(
    tests_ph3, 'Phase 3', 'PH3 Products OPTIONS - 503 before cache load',
    status, 503, ms
)

status, hdrs, ms = cors_options_request(
    endpoint_url, ph3_analytics_path, 'https://dashboard.contoso.com'
)
track(
    tests_ph3, 'Phase 3', 'PH3 Analytics OPTIONS - 503 before cache load',
    status, 503, ms
)


# ---- Step 3: Load the cache via admin API ----

print('\nStep 3: Loading CORS origin cache via admin API ...\n')

load_url = f'{endpoint_url}/{ph3_admin_path}/load-cache/{cors_cache_key}'
load_headers = {**admin_base_headers, 'Content-Type': 'application/json'}
load_resp = http_session.post(
    load_url,
    headers = load_headers,
    data = cors_cache_mapping_json,
    timeout = 30,
)
load_ms = round(load_resp.elapsed.total_seconds() * 1000, 1)

track(
    tests_ph3, 'Phase 3', 'PH3 Admin POST /load-cache - 200 OK',
    load_resp.status_code, 200, load_ms
)

load_body = load_resp.json()
track(
    tests_ph3, 'Phase 3', 'PH3 Admin load response - correct cache key',
    load_body.get('cacheKey'), cors_cache_key
)

print(f'  Load response: {json.dumps(load_body, indent=2)}')


# ---- Step 4: Run standard CORS tests (same matrix as Phases 1 & 2) ----

print('\nStep 4: Running CORS tests after cache load ...\n')

run_phase_tests(tests_ph3, 'Phase 3', ph3_products_path, ph3_analytics_path)
tests_ph3.print_summary()

print_ok('Phase 3 tests complete - cache-backed per-API CORS verified')

### 📊 Consolidated Test Results

Aggregates the results from all four test phases into a single summary table.

In [ ]:
import pandas as pd
from IPython.display import display
from console import print_ok

if 'test_results' not in locals() or not test_results:
    raise SystemExit(1)

df = pd.DataFrame(test_results)
df.index = range(1, len(df) + 1)
df.index.name = '#'
df['Test'] = df['Test'].str.replace(
    r'^(?:BL|PHASE\d|PH3)\s+', '', regex=True
)
df['Result'] = df['Result'].map({True: '✅ Pass', False: '❌ Fail'})
df['Duration (ms)'] = df['Duration (ms)'].apply(
    lambda v: f'{v:.1f}' if pd.notna(v) else 'N/A'
)

passed = (df['Result'] == '✅ Pass').sum()
failed = (df['Result'] == '❌ Fail').sum()
total = len(df)

# Phase colour bands - each uses explicit dark text for theme safety.
# All contrasts verified against WCAG 2.0 AA (>= 4.5:1 for normal text).
#   Warm-up   #e5e7eb  text #111827  ~ 14.3:1
#   Baseline  #dbeafe  text #1e3a5f  ~  7.2:1
#   Phase 1   #fef3c7  text #78350f  ~  6.8:1
#   Phase 2   #ede9fe  text #4c1d95  ~  7.5:1
#   Phase 3   #fce7f3  text #831843  ~  7.1:1
#   Pass      #d1fae5  text #065f46  ~  5.8:1
#   Fail      #fee2e2  text #7f1d1d  ~  6.5:1
phase_styles = {
    'Warm-up':  ('background-color: #e5e7eb; color: #111827'),
    'Baseline': ('background-color: #dbeafe; color: #1e3a5f'),
    'Phase 1':  ('background-color: #fef3c7; color: #78350f'),
    'Phase 2':  ('background-color: #ede9fe; color: #4c1d95'),
    'Phase 3':  ('background-color: #fce7f3; color: #831843'),
}
STYLE_PASS = (
    'background-color: #d1fae5; color: #065f46; font-weight: 600'
)
STYLE_FAIL = (
    'background-color: #fee2e2; color: #7f1d1d; font-weight: 600'
)


def highlight_row(row):
    """Colour the entire row by phase + highlight the Result cell."""
    phase_css = phase_styles.get(row['Phase'], 'color: #212529')
    styles = [phase_css] * len(row)
    result_idx = row.index.get_loc('Result')
    if '✅' in str(row['Result']):
        styles[result_idx] = STYLE_PASS
    elif '❌' in str(row['Result']):
        styles[result_idx] = STYLE_FAIL
    return styles


styled = (
    df.style
    .apply(highlight_row, axis=1)
    .set_properties(**{
        'text-align': 'left',
        'border': '1px solid #d1d5db',
        'padding': '6px 10px',
    })
    .set_properties(
        subset=['Duration (ms)'],
        **{'text-align': 'right', 'font-variant-numeric': 'tabular-nums'},
    )
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                # #1e3a5f on white = 8.5:1; white on #1e3a5f = 8.5:1
                ('background-color', '#1e3a5f'),
                ('color', 'white'),
                ('padding', '10px 12px'),
                ('text-align', 'left'),
                ('border', '1px solid #d1d5db'),
                ('font-weight', '600'),
            ],
        },
        {
            'selector': 'caption',
            'props': [
                ('font-size', '15px'),
                ('font-weight', 'bold'),
                ('padding', '10px 0'),
                # #1e3a5f on white = 8.5:1
                ('color', '#1e3a5f'),
            ],
        },
        {
            'selector': 'td',
            'props': [
                ('border-bottom', '1px solid #e5e7eb'),
            ],
        },
    ])
    .set_caption(
        f'Total: {total}  |  Passed: {passed}  |  Failed: {failed}'
    )
)

display(styled)

print_ok('All done!')